# Pre-class: Machine learning interatomic potentials (MLIP) basics

**MS.49900 AI for Materials Science, Hands-on session 7 (pre-class)**

Finish this notebook before class (about 60 min). In class we will compare several universal MLIPs and fine-tune one.

1. Set up Colab and SevenNet
2. ASE basics: `Atoms` and `Calculator`
3. Practice: predict DFT energies, forces and stress of Li₆PS₅Cl with SevenNet-0
4. Question for class

Cells marked `# You can change here` are blanks you fill in.

## 1. Setup
Use a GPU runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU. Runtime > Change runtime type > T4 GPU, then run again from the top.')

In [ ]:
# Install SevenNet and download the course repository with the Li6PS5Cl DFT data (about 1 min)
!pip install -q sevenn==0.13.0

import os
if not os.path.exists('MS49900-AI4M'):
    !git clone -q https://github.com/kwongibaek/MS49900-AI4M.git

DATA = 'MS49900-AI4M/Data'
print(sorted(f for f in os.listdir(DATA) if f.startswith(('lpsc_', 'sevennet_'))))

In [ ]:
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ase.io import read
from ase.units import bar
from sevenn.calculator import SevenNetCalculator

to_kbar = 1 / bar / 1000          # eV/Å³ -> kbar


def dft_labels(traj):
    """DFT energy (eV/atom), forces (eV/Å) and stress (kbar) stored in the data file."""
    E = np.array([a.get_potential_energy() / len(a) for a in traj])
    F = np.concatenate([a.get_forces().ravel() for a in traj])
    S = np.concatenate([a.get_stress().ravel() for a in traj]) * to_kbar
    return E, F, S


def predict(calc, traj):
    """MLIP energy, forces and stress for every structure, plus seconds per structure."""
    E, F, S = [], [], []
    warm = traj[0].copy(); warm.calc = calc; warm.get_forces()   # warm-up, not timed
    t0 = time.time()
    for atoms in traj:
        a = atoms.copy()          # copy keeps the DFT labels of the original
        a.calc = calc
        E.append(a.get_potential_energy() / len(a))
        F.append(a.get_forces().ravel())
        S.append(a.get_stress().ravel() * to_kbar)
    sec = (time.time() - t0) / len(traj)
    return np.array(E), np.concatenate(F), np.concatenate(S), sec


def free_gpu(calc=None):
    del calc
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 2. ASE basics

[ASE](https://wiki.fysik.dtu.dk/ase/) stores a structure as an `Atoms` object.
A **calculator** attached to `Atoms` returns energy, forces and stress. DFT codes and MLIPs both work as calculators, so the same simulation code runs with either.

Our system is **Li₆PS₅Cl** (argyrodite), a solid electrolyte for all-solid-state batteries. The data are DFT MD snapshots of a 2×2×2 supercell (416 atoms) at 600 K and 1200 K.

In [ ]:
atoms = read(f'{DATA}/lpsc_600K.extxyz.gz', index=0)
print('Formula :', atoms.get_chemical_formula())
print('Atoms   :', len(atoms))
print('Volume  :', round(atoms.get_volume(), 1), 'Å³ (2x2x2 supercell)')

# The file also stores DFT results for each snapshot
print('DFT energy :', round(atoms.get_potential_energy() / len(atoms), 4), 'eV/atom')
print('DFT forces :', atoms.get_forces().shape, 'eV/Å')

In [ ]:
from ase.visualize import view
view(atoms, viewer='x3d')

Attach a pretrained SevenNet model as the calculator. `7net-0` was trained on the Materials Project relaxation data (MPtrj).

In [ ]:
calc = SevenNetCalculator('7net-0')

mlip_atoms = atoms.copy()      # copy without the DFT results
mlip_atoms.calc = calc
print('7net-0 energy :', round(mlip_atoms.get_potential_energy() / len(mlip_atoms), 4), 'eV/atom')
print('7net-0 forces :', mlip_atoms.get_forces().shape)
print('7net-0 stress :', (mlip_atoms.get_stress() * to_kbar).round(2), 'kbar')

## 3. Practice: SevenNet-0 vs DFT

Use 10% of the 600 K snapshots, predict them with 7net-0 and compare with DFT.

In [ ]:
full_traj = read(f'{DATA}/lpsc_600K.extxyz.gz', ':')
rng = np.random.default_rng(42)
idx = np.sort(rng.choice(len(full_traj), int(0.1 * len(full_traj)), replace=False))
traj = [full_traj[i] for i in idx]
print(f'{len(traj)} of {len(full_traj)} structures')

dft_E, dft_F, dft_S = dft_labels(traj)

**Blank 1.** Fill the prediction loop.

In [ ]:
E, F, S = [], [], []
for atoms in traj:
    atoms = atoms.copy()
    atoms.calc = calc
# You can change here------------------------------------------------------------
    E.append(...)   # energy per atom (eV/atom)
    F.append(...)   # forces, flattened (eV/Å)
    S.append(...)   # stress in kbar (multiply by to_kbar)
# -------------------------------------------------------------------------------

mlip_E, mlip_F, mlip_S = np.array(E), np.concatenate(F), np.concatenate(S)
print(len(mlip_E), 'energies,', len(mlip_F), 'force components,', len(mlip_S), 'stress components')

**Blank 2.** Write RMSE and MAE.

In [ ]:
# You can change here------------------------------------------------------------
def rmse(y_true, y_pred):
    return ...   # root mean squared error

def mae(y_true, y_pred):
    return ...   # mean absolute error
# -------------------------------------------------------------------------------

table = pd.DataFrame({
    'RMSE': [rmse(dft_E, mlip_E), rmse(dft_F, mlip_F), rmse(dft_S, mlip_S)],
    'MAE':  [mae(dft_E, mlip_E),  mae(dft_F, mlip_F),  mae(dft_S, mlip_S)],
}, index=['Energy (eV/atom)', 'Force (eV/Å)', 'Stress (kbar)'])
table.round(4)

### Parity plots

Absolute energies span a very small range, so the energy plot uses **relative energies** (each set minus its own mean).

In [ ]:
def parity(ax, x, y, label, unit):
    ax.scatter(x, y, s=3, alpha=0.3)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_xlabel(f'DFT {label} ({unit})')
    ax.set_ylabel(f'MLIP {label} ({unit})')
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
parity(axes[0], dft_E - dft_E.mean(), mlip_E - mlip_E.mean(), 'relative energy', 'eV/atom')
parity(axes[1], dft_F, mlip_F, 'force', 'eV/Å')
parity(axes[2], dft_S, mlip_S, 'stress', 'kbar')
fig.suptitle('7net-0 vs DFT, Li₆PS₅Cl 600 K')
plt.tight_layout(); plt.show()

print('Mean energy  DFT:', dft_E.mean().round(3), ' 7net-0:', mlip_E.mean().round(3), 'eV/atom')

## 4. Question for class

The energy RMSE and MAE are almost the same (about 2.4 eV/atom), but the relative energies follow DFT.

**Q.** What does this tell you about the energy error? Why could a model trained on Materials Project data show this error on this dataset? Write 2–3 sentences below. We start the class with this question.

*Your answer:*

## Optional: other SevenNet pretrained models

`SevenNetCalculator` accepts other pretrained models by name. Some multi-task models need `modal` (which DFT setting to reproduce). We compare them in class.

| Name | Training data | Note |
|---|---|---|
| `7net-0` | MPtrj | small and fast |
| `7net-l3i5` | MPtrj | higher order, slower |
| `7net-mf-ompa` | MPtrj + sAlex + OMat24 | `modal='mpa'` or `'omat24'` |
| `7net-omat` | OMat24 | |
| `7net-omni` | 15 open DFT datasets | `modal='mpa'`, `'matpes_pbe'`, ... |